In [1]:
import torch
# torch.autograd.set_detect_anomaly(True)
from torch.utils.data import DataLoader,WeightedRandomSampler
from torch.optim.lr_scheduler import PolynomialLR
import torch.nn.functional as F
from torchvision.transforms import v2
import pandas as pd
import albumentations as A
# from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm.notebook import tqdm
import json
import cv2
import torch.nn as nn
# import matplotlib
# matplotlib.use("Agg")
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import zarr
import gc
%load_ext autoreload
%autoreload 2
%matplotlib inline
###IE###
from utils.helpers import (
    read_images,load_model,bootstrap_model,generate_all_masks,
    make_result_dataloaders,make_training_dataloaders,
    draw_fast_plots,TP_TN_FP_FN,read_yolo_bboxes
)
from utils.preprocessing import (
    normalize_xca,normalize_empty,get_preprocessing,
    remove_black_border_multithread,BrightnessMultiplicativeNNUNet2D,
    ContrastAugmentationNNUNet2D
)
from utils.draw_model_feature_maps import draw_feature_maps
from utils.draw_loss_maps import draw_loss_plots
from utils.model_aug_analysis import  aug_analysis
from utils.calculate_classwise_capture import classwise_capture
# from utils.debuging import show_bootstraping_results
from models.nnunet import nnUnet
from models.memory_unet import MemoryUnet
from models.swin_encoder import SwinUperNet
from models.hrnet import HRNET
from models.moe_model import MoeUnet
from utils.testmodel import UTANet
from utils.losses import UnetLoss
from utils.recorder import HistoryRecorder
from logger import save_full_report
from trainer import trainer
from trainer import evaluation
from image_clusters import valid,problem_maps
from utils.graph_extraction import (
    create_gnn_dataset,generate_robust_mask,
    create_gnn_dataset_parallel_limited)
from models.Swin_Unet.networks.vision_transformer import  SwinUnet
from utils.debuging import weighting_logic , save_loss_weights , save_weight_overlay_cv2
###SS###

/home/parsa/Venvs/AI/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model_name = "2026-07-14_18-47-22 [multi-right-9class-150epo-aux_branchpoint-nobias-cbam-upsample]"
out_folder = f"./outputs/{model_name}"
best_model,args,train_transforms = load_model(
    model_class=MoeUnet,
    path=out_folder,
    model_inputs={},
    device="cuda",
    return_args = True
)
best_model.eval()
right_hard_classes = [20,21,22,23]
# losses_keys = ["total loss","FCE loss",args["loss_type"]]
losses_keys = ["total loss",args["loss_type"],"mask CE-Loss"]#,"cl_dice_loss","mask CE-Loss" "branch_"+args["loss_type"],"branch_mask CE-Loss"
if(args["deep_super_vision"]):
    shape = args["image_shape"]
    loss_weights = [1/(2**i) for i in range(1,len(args["moe_unet"]["encoder_out_channels"]))]
else:
    loss_weights = []
loss_weights


[0.5, 0.25, 0.125, 0.0625]

In [3]:
# train_df = pd.read_csv(f"./dataset/syntax/model_analysis/{model_name}/aug_analysis/train_aug_analysis.csv")
# test_df = pd.read_csv(f"./dataset/syntax/model_analysis/{model_name}/aug_analysis/test_aug_analysis.csv")
# val_df = pd.read_csv(f"./dataset/syntax/model_analysis/{model_name}/aug_analysis/val_aug_analysis.csv")

# print("train")
# display(train_df.drop(columns=[name for name in train_df.columns if "_d" not in name]+["names"]).mean(axis=0).sort_values())
# print("val")
# display(val_df.drop(columns=[name for name in train_df.columns if "_d" not in name]+["names"]).mean(axis=0).sort_values())
# print("test")
# display(test_df.drop(columns=[name for name in train_df.columns if "_d" not in name]+["names"]).mean(axis=0).sort_values())

In [4]:
_ , test_transforms = get_preprocessing(args["training_mode"],args)

if(args["in_c"]==1):
    if(args["training_mode"]!="cleaning"):
        normalizer = normalize_xca
        print("set to z score")
    else:
        normalizer = normalize_empty
elif(args["in_c"]==2):
    print("set to z score 2c")
    normalizer = normalize_empty
else:
    normalizer = A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
    )
    print("set to imagenet")


set to z score


In [5]:
train_images,sampler_weights,last_boot_level = read_images(
    paths = args["paths"],
    part = "train",
    chosen_labels=None,
    hard_classes = right_hard_classes,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"]

)
valid_images = read_images(
    paths = args["paths"],
    part = "val",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"]
)
test_images = read_images(
    paths = args["paths"],
    part = "test",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"]
)

0


  0%|          | 0/1000 [00:00<?, ?it/s]

0


  0%|          | 0/200 [00:00<?, ?it/s]

0


  0%|          | 0/300 [00:00<?, ?it/s]

In [6]:
if(args["training_mode"]=="full"):
    b=0.999

    train_class_counts = [
        1200,374,375,369,303,525,537,
        198,340,70,21,310,1,61,320,129,
        63,305,107,49,127,38,232,43,48,31
    ]

    total = np.sum(train_class_counts)
    mask_ce_weights = np.log(total/np.array(train_class_counts))
    mask_ce_weights = (mask_ce_weights / mask_ce_weights.mean()).tolist()
    mask_ce_weights[0]=0.1

    args["mask_ce_weights"]=mask_ce_weights
    args["mask_ce_weights"]
elif(args["training_mode"]=="cleaning"):
    args["mask_ce_weights"] = [0.1,1]
else:
    # class_counts = np.zeros(args["class_count"])
    # for img,hard_idx,label,skel_mask,side,name_stem in train_images:
    #     u_idx = np.unique(label)
    #     class_counts[u_idx]+=1
    # alpha = (1.0 / class_counts)
    # alpha = alpha / alpha.sum() * len(class_counts)
    # args["fce_conf"]["alpha"] = alpha.tolist()

    class_counts = np.zeros(args["class_count"])
    for img,hard_idx,label,skel_mask,side,name_stem in train_images:
        u_idx = np.unique(label)
        counts = np.bincount(label.reshape(-1))
        for u in u_idx :
            class_counts[u]+=counts[u]
    alpha = (1.0 / class_counts)
    alpha = alpha / alpha.sum() * len(class_counts)
    alpha[0]=np.min(alpha[1:])*0.5
    # alpha[0] = 0.01
    args["fce_conf"]["alpha"] = alpha.tolist()
    # args["tversky_conf"]["weights"] = alpha[1:].tolist()
    print(f"weights are set to : {alpha}")
    # pass

weights are set to : [0.08968223 0.18538219 0.17936446 0.18607616 0.25631113 0.38241285
 2.49209757 1.93844191 3.37862178]


In [7]:
sampler = WeightedRandomSampler(
    weights=sampler_weights, 
    num_samples=len(sampler_weights),
    replacement=True
)
train_loader , valid_loader , test_loader = make_training_dataloaders(
    train_images=train_images,
    valid_images=valid_images,
    test_images=test_images,
    normalizer=normalizer,
    train_transforms=train_transforms,
    test_transforms=test_transforms,
    args=args,
    sampler=None
)
re_train_loader , re_valid_loader , re_test_loader  = make_result_dataloaders(
    train_images=train_images,
    valid_images=valid_images,
    test_images=test_images,
    normalizer=normalizer,
    transforms=test_transforms
)

In [8]:
data_loaders = {
    "train": re_train_loader,
    "val": re_valid_loader,
    "test": re_test_loader
}
labels_base = Path(args["paths"]["base_path"])/args["paths"]["labels_folder"]





In [9]:
# data_loaders = {
#     "train": re_train_loader,
#     "val": re_valid_loader,
#     "test": re_test_loader,
# }
# ratios = {
#     "train":[],
#     "val" :[],
#     "test":[]
# }
# for part,dl in data_loaders.items():
#     for data in dl:
#         mask = (data["mask"]!=0).int()
#         fg = torch.sum(mask)
#         ratio = fg/(512*512)
#         ratio = ratio.item()
#         ratios[part].append(ratio)

# ratios = {part:np.mean(items) for part,items in ratios.items()}
# ratios

In [10]:
"""
{'train': np.float64(0.032682949066162106),
 'val': np.float64(0.035836372375488285),
 'test': np.float64(0.03582077026367188)}
"""

"\n{'train': np.float64(0.032682949066162106),\n 'val': np.float64(0.035836372375488285),\n 'test': np.float64(0.03582077026367188)}\n"

In [11]:
best_model

MoeUnet(
  (MainEncoder): Encoder(
    (base_conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): ReLU(inplace=True)
      (3): Dropout2d(p=0.5, inplace=False)
    )
    (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (model): ModuleDict(
      (encoder_block_[1]): EncoderBlocks(
        (start_conv): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
          (2): ReLU()
          (3): Dropout2d(p=0.5, inplace=False)
          (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (5): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
          (6): CBAM_BLOCK(
            (maxpool): AdaptiveMaxPool2

In [12]:
class GeneralizedDiceLoss(nn.Module):
    def __init__(self, eps=1e-6, sum_dims=(0, 2, 3)):
        super().__init__()
        self.eps = eps
        self.sum_dims = sum_dims

    def forward(self, pred_probs, gt):
        gt = gt.float()
        true_volume = gt.sum(dim=self.sum_dims)
        weights = 1.0 / (true_volume ** 2 + self.eps)

        # empty classes get a 1/eps spike -> replace with max present weight
        present = true_volume > 0
        if present.any():
            weights = torch.where(present, weights, weights[present].max())
        else:
            weights = torch.zeros_like(weights)

        intersection = (pred_probs * gt).sum(dim=self.sum_dims)
        denominator  = (pred_probs + gt).sum(dim=self.sum_dims)

        num = 2.0 * (weights * intersection).sum() + self.eps
        den = (weights * denominator).sum() + self.eps
        return 1.0 - num / den

In [13]:


best_model.eval()
device = "cuda"
ACT_MODE = "mean"
# layers_to_print = {
#     "base_conv": best_model.MainEncoder.base_conv,
#     "E1": best_model.MainEncoder.model["encoder_block_[1]"],
#     "E2": best_model.MainEncoder.model["encoder_block_[2]"],
#     "E3": best_model.MainEncoder.model["encoder_block_[3]"],
#     # "E4": best_model.MainEncoder.model["encoder_block_[4]"],
#     # "E5": best_model.MainEncoder.model["encoder_block_[5]"],
#     # "E6": best_model.MainEncoder.model["encoder_block_[6]"],
#     # "E7": best_model.MainEncoder.model["encoder_block_[7]"],
#     "E1_CBAM_Spatial": best_model.MainEncoder.model["encoder_block_[1]"].enc_convs["enc_conv_(0)"][5].spatial_sigmoid,
#     "E2_CBAM_Spatial": best_model.MainEncoder.model["encoder_block_[2]"].enc_convs["enc_conv_(0)"][5].spatial_sigmoid,
#     "E3_CBAM_Spatial": best_model.MainEncoder.model["encoder_block_[3]"].enc_convs["enc_conv_(0)"][5].spatial_sigmoid,
#     # "E4_CBAM_Spatial": best_model.MainEncoder.model["encoder_block_[4]"].enc_convs["enc_conv_(0)"][5].spatial_sigmoid,
#     # "E5_CBAM_Spatial": best_model.MainEncoder.model["encoder_block_[5]"].enc_convs["enc_conv_(0)"][5].spatial_sigmoid,
#     # "E6_CBAM_Spatial": best_model.MainEncoder.model["encoder_block_[6]"].enc_convs["enc_conv_(0)"][5].spatial_sigmoid,
#     # "E7_CBAM_Spatial": best_model.MainEncoder.model["encoder_block_[7]"].enc_convs["enc_conv_(0)"][5].spatial_sigmoid,
#     # "D7": best_model.MainDecoder.model["decoder_block_[7]"],
#     # "D6": best_model.MainDecoder.model["decoder_block_[6]"],
#     # "D5": best_model.MainDecoder.model["decoder_block_[5]"],
#     # "D4": best_model.MainDecoder.model["decoder_block_[4]"],
#     "D3": best_model.MainDecoder.model["decoder_block_[3]"],
#     "D2": best_model.MainDecoder.model["decoder_block_[2]"],
#     "D1": best_model.MainDecoder.model["decoder_block_[1]"],
#     # "D7_CBAM_Spatial": best_model.MainDecoder.model["decoder_block_[7]"].cbam_block.spatial_sigmoid,
#     # "D6_CBAM_Spatial": best_model.MainDecoder.model["decoder_block_[6]"].cbam_block.spatial_sigmoid,
#     # "D5_CBAM_Spatial": best_model.MainDecoder.model["decoder_block_[5]"].cbam_block.spatial_sigmoid,
#     # "D4_CBAM_Spatial": best_model.MainDecoder.model["decoder_block_[4]"].cbam_block.spatial_sigmoid,
#     "D3_CBAM_Spatial": best_model.MainDecoder.model["decoder_block_[3]"].cbam_block.spatial_sigmoid,
#     "D2_CBAM_Spatial": best_model.MainDecoder.model["decoder_block_[2]"].cbam_block.spatial_sigmoid,
#     "D1_CBAM_Spatial": best_model.MainDecoder.model["decoder_block_[1]"].cbam_block.spatial_sigmoid,
# }
# # layer_names = [
# #     "base_conv",
# #     "E1", "E2", "E3", "E4", "E5","E6","E7","E1_CBAM_Spatial",
# #     "E2_CBAM_Spatial","E3_CBAM_Spatial","E4_CBAM_Spatial", 
# #     "E5_CBAM_Spatial","E6_CBAM_Spatial","E7_CBAM_Spatial",
# #     "D7","D6","D5", "D4", "D3", "D2", "D1",
# #     "D7_CBAM_Spatial","D6_CBAM_Spatial","D5_CBAM_Spatial",
# #     "D4_CBAM_Spatial","D3_CBAM_Spatial","D2_CBAM_Spatial","D1_CBAM_Spatial",
# # ]
# layer_names = list(layers_to_print.keys())
data_loaders = {
    "train": re_train_loader,
    "val": re_valid_loader,
    "test": re_test_loader,
}



dict_images = {
    "train": train_images,
    "val" : valid_images,
    "test":test_images
}
dice_loss_fn = GeneralizedDiceLoss(eps=1e-8,sum_dims=(0,2,3))
# draw_feature_maps(
#     layers_to_print,
#     layer_names,
#     data_loaders,
#     args,
#     model_name,
#     best_model,
#     ACT_MODE,
#     device,
#     num_workers=8,
#     out_folder="model_analysis"
# )

# draw_loss_plots(
#     model=best_model,
#     args=args,
#     dataloaders = data_loaders,
#     num_workers=8,
#     model_name=model_name,
#     out_folder="model_analysis"
# )

aug_analysis(
    best_model=best_model,
    images=dict_images,
    args = args,
    normalizer=normalizer,
    model_name=model_name,
    dice_loss_fn= dice_loss_fn,
    out_folder="model_analysis"
)
# classwise_capture(
#     best_model=best_model,
#     data_loaders=data_loaders,
#     model_name=model_name,
#     args=args,
#     out_folder="model_analysis"
# )

  0%|          | 0/375 [00:00<?, ?it/s]

  0%|          | 0/66 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
best_model

MoeUnet(
  (MainEncoder): Encoder(
    (base_conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): ReLU(inplace=True)
      (3): Dropout2d(p=0.5, inplace=False)
    )
    (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (model): ModuleDict(
      (encoder_block_[1]): EncoderBlocks(
        (start_conv): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
          (2): ReLU()
          (3): Dropout2d(p=0.5, inplace=False)
          (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (5): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
          (6): CBAM_BLOCK(
            (maxpool): AdaptiveMaxPool2